In [1]:
from pathlib import Path
import pandas as pd
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from html import escape

# ---------------------------------------------------------
# 1. Arquivos
# ---------------------------------------------------------
arquivo_entrada = Path("08_RELATORIO_CLASSIFICACAO_VARIAVEIS.csv")
nome_base_saida = "09_CLASSIFICACAO_MANUAL_VARIAVEIS"

if not arquivo_entrada.exists():
    raise FileNotFoundError(
        f"Não encontrei {arquivo_entrada.resolve()}. "
        "Coloque o CSV na mesma pasta deste notebook."
    )

# Lê o CSV aceitando separador ; ou ,.
try:
    relatorio = pd.read_csv(arquivo_entrada, sep=";", encoding="utf-8-sig", dtype=str)
    if relatorio.shape[1] == 1:
        relatorio = pd.read_csv(arquivo_entrada, sep=",", encoding="utf-8-sig", dtype=str)
except UnicodeDecodeError:
    relatorio = pd.read_csv(arquivo_entrada, sep=";", encoding="latin-1", dtype=str)
relatorio = relatorio.fillna("")

# ---------------------------------------------------------
# 2. Remover colunas que não devem aparecer na tabela
# ---------------------------------------------------------
colunas_remover = {
    "tipo pandas",
    "não nulo", "não nulos", "nao nulo", "nao nulos",
    "nulo", "nulos",
    "valores distintos",
    "valores observados", "valores observados (amostra)",
    "classificação manual da variável", "classificacao manual da variavel",
    "tipo de variável", "tipo de variavel",
}
base = relatorio[
    [c for c in relatorio.columns if c.strip().casefold() not in colunas_remover]
].copy()

# Opções de tipos Pandas/NumPy e tipos especializados comuns.
OPCOES_TIPO_PANDAS = [
    "object", "string", "category",
    "bool", "boolean",
    "int8", "int16", "int32", "int64",
    "uint8", "uint16", "uint32", "uint64",
    "Int8", "Int16", "Int32", "Int64",
    "UInt8", "UInt16", "UInt32", "UInt64",
    "float16", "float32", "float64",
    "Float32", "Float64",
    "datetime64[ns]", "datetime64[ns, UTC]", "timedelta64[ns]",
]
COL_DTYPE = "Tipo de variável"
dtypes_existentes = (
    relatorio[COL_DTYPE].tolist() if COL_DTYPE in relatorio.columns else [""] * len(base)
)

print(f"Arquivo carregado: {arquivo_entrada.name}")
print(f"Variáveis para classificar: {len(base)}")

# Tabela: colunas existentes + um único dropdown para Tipo de variável.
num_cols = len(base.columns) + 1
grid = widgets.GridspecLayout(len(base) + 1, num_cols, width="100%")
base_cols = list(base.columns)

def largura_coluna(col):
    nome = str(col).strip().casefold()
    if nome in {"código", "codigo", "códigos", "codigos"}:
        return "minmax(55px, 0.35fr)"
    if nome in {"variável", "variavel", "descrição", "descricao"}:
        return "minmax(400px, 3.5fr)"
    return "minmax(85px, 0.7fr)"

grid.layout.grid_template_columns = (
    " ".join(largura_coluna(col) for col in base_cols)
    + " minmax(190px, 1fr)"
)
for j, col in enumerate(base_cols):
    grid[0, j] = widgets.HTML(value=f"<b>{escape(str(col))}</b>")
grid[0, len(base_cols)] = widgets.HTML(value=f"<b>{COL_DTYPE}</b>")

widgets_dtype = []
for i in range(len(base)):
    for j, col in enumerate(base_cols):
        valor_html = escape(str(base.iloc[i, j]))
        grid[i + 1, j] = widgets.HTML(
            value=f"<div style='white-space:normal; overflow-wrap:anywhere'>{valor_html}</div>"
        )

    dtype_atual = dtypes_existentes[i] if i < len(dtypes_existentes) else ""
    if dtype_atual not in OPCOES_TIPO_PANDAS:
        dtype_atual = ""
    w_dtype = widgets.Dropdown(
        options=[("— selecione —", "")] + [(x, x) for x in OPCOES_TIPO_PANDAS],
        value=dtype_atual,
        description="",
        layout=widgets.Layout(width="100%"),
    )
    widgets_dtype.append(w_dtype)
    grid[i + 1, len(base_cols)] = w_dtype

area_tabela = widgets.Box(
    [grid],
    layout=widgets.Layout(overflow="auto", max_height="700px", width="100%")
)
display(area_tabela)

# ---------------------------------------------------------
# 3. Gravar somente quando todos os tipos estiverem preenchidos
# ---------------------------------------------------------
saida_mensagem = widgets.Output()
botao_salvar = widgets.Button(
    description="Salvar classificação",
    button_style="success",
    icon="save",
)

def salvar_classificacoes(_):
    with saida_mensagem:
        clear_output()
        dtypes = [w.value for w in widgets_dtype]
        pendentes = sum(not v for v in dtypes)
        if pendentes:
            print("ARQUIVO NÃO SALVO: ainda há tipos de variável pendentes.")
            print(f"Variáveis sem tipo selecionado: {pendentes}")
            print("Preencha o menu Tipo de variável de todas as linhas e tente novamente.")
            return

        resultado = base.copy()
        resultado[COL_DTYPE] = dtypes
        agora = datetime.now()
        # Nome seguro para arquivos: / e : não podem ser usados como separadores
        # no nome do arquivo. Usamos hífens mantendo o formato DD-MMM_HH-MM-SS.
        meses_pt = ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun", "Jul", "Ago", "Set", "Out", "Nov", "Dez"]
        data_hora = f"{agora.day:02d}-{meses_pt[agora.month - 1]}_{agora:%H-%M-%S}"
        arquivo_saida = Path(f"09_class_manual_variaveis_{data_hora}.csv")
        resultado.to_csv(arquivo_saida, sep=";", index=False, encoding="utf-8-sig")
        print("Classificação gravada com sucesso:")
        print(arquivo_saida.resolve())
        print(f"Total de variáveis classificadas: {len(resultado)}")
        display(HTML(f"<b>Salvo com sucesso:</b> {escape(arquivo_saida.name)}"))

botao_salvar.on_click(salvar_classificacoes)
display(widgets.HBox([botao_salvar]), saida_mensagem)


Arquivo carregado: 08_RELATORIO_CLASSIFICACAO_VARIAVEIS.csv
Variáveis para classificar: 41


Box(children=(GridspecLayout(children=(HTML(value='<b>Código</b>', layout=Layout(grid_area='widget001')), HTML…

Output()